<a href="https://colab.research.google.com/github/karnika-soni/Indic-Multimodal-NMT/blob/main/multimodal_mBART_mode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [ ]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(
    MODEL_NAME
)

model = MBartForConditionalGeneration.from_pretrained(
    MODEL_NAME
)

model.to(device)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
print(model)

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [ ]:
print(model.config.d_model)
print(model.config.encoder_layers)
print(model.config.decoder_layers)
print(model.config.vocab_size)

1024
12
12
250054


- Detectron2 Features
        │
        ▼
 - Vision Encoder
        │
        ▼
 - Visual Embeddings

In [ ]:
import torch
import torch.nn as nn


class VisionEncoder(nn.Module):

    def __init__(self, hidden_size=1024):

        super().__init__()

        self.layer_norm = nn.LayerNorm(hidden_size)

        self.dropout = nn.Dropout(0.1)

    def forward(self, image_features):

        image_features = self.layer_norm(image_features)

        image_features = self.dropout(image_features)

        return image_features

In [ ]:
vision_encoder = VisionEncoder().to(device)

In [ ]:
dummy = torch.randn(
    8,
    36,
    1024
).to(device)

output = vision_encoder(dummy)

print(output.shape)

torch.Size([8, 36, 1024])


In [ ]:
class LateFusion(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(
        self,
        text_features,
        vision_features
    ):

        fused_features = torch.cat(
            [
                text_features,
                vision_features
            ],
            dim=1
        )

        return fused_features

In [ ]:
batch_size = 4

text_features = torch.randn(
    batch_size,
    20,
    1024
).to(device)


vision_features = torch.randn(
    batch_size,
    36,
    1024
).to(device)

In [ ]:
fusion = LateFusion()

fused = fusion(
    text_features,
    vision_features
)

print(fused.shape)

torch.Size([4, 56, 1024])


In [ ]:
from transformers.modeling_outputs import BaseModelOutput

class MultimodalMBart(nn.Module):

    def __init__(self, mbart_model):

        super().__init__()

        self.mbart = mbart_model

        self.vision_encoder = VisionEncoder()

        self.fusion = LateFusion()


    def forward(
        self,
        input_ids,
        attention_mask,
        image_features,
        labels=None
    ):

        # -------------------------
        # 1. Text encoding
        # -------------------------

        encoder_outputs = self.mbart.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )


        text_hidden = (
            encoder_outputs.last_hidden_state
        )


        # -------------------------
        # 2. Image encoding
        # -------------------------

        vision_hidden = self.vision_encoder(
            image_features
        )


        # -------------------------
        # 3. Fusion
        # -------------------------

        fused_hidden = self.fusion(
            text_hidden,
            vision_hidden
        )


        vision_mask = torch.ones(
            vision_hidden.size()[:2],
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )


        fused_attention_mask = torch.cat(
            [
                attention_mask,
                vision_mask
            ],
            dim=1
        )


        # -------------------------
        # 4. Decoder
        # -------------------------


        encoder_outputs = BaseModelOutput(
            last_hidden_state=fused_hidden
        )


        outputs = self.mbart(
            encoder_outputs=encoder_outputs,
            attention_mask=fused_attention_mask,
            labels=labels
        )

        return outputs

    def encode_multimodal(
        self,
        input_ids,
        attention_mask,
        image_features,
    ):
        # -------------------------
        # 1. Text encoding
        # -------------------------
        text_encoder_outputs = self.mbart.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        text_hidden = text_encoder_outputs.last_hidden_state

        # -------------------------
        # 2. Image encoding
        # -------------------------
        vision_hidden = self.vision_encoder(
            image_features
        )

        # -------------------------
        # 3. Fusion
        # -------------------------
        fused_hidden = self.fusion(
            text_hidden,
            vision_hidden
        )

        # -------------------------
        # 4. Attention mask
        # -------------------------
        vision_mask = torch.ones(
            vision_hidden.size()[:2],
            dtype=attention_mask.dtype,
            device=attention_mask.device
        )

        fused_attention_mask = torch.cat(
            [
                attention_mask,
                vision_mask
            ],
            dim=1
        )

        return (
            BaseModelOutput(
                last_hidden_state=fused_hidden
            ),
            fused_attention_mask
        )

    @torch.no_grad()
    def generate(
        self,
        input_ids,
        attention_mask,
        image_features,
        **generate_kwargs
    ):

        encoder_outputs, fused_attention_mask = \
            self.encode_multimodal(
                input_ids,
                attention_mask,
                image_features
            )

        generated_ids = self.mbart.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=fused_attention_mask,
            **generate_kwargs
        )

        return generated_ids

In [ ]:
# The line `vision_mask = torch.ones(vision_hidden.size()[:2], dtype=attention_mask.dtype, device=attention_mask.device)` creates a new PyTorch tensor named `vision_mask`.
#
# Here's a breakdown of what each part does:
#
# *   `torch.ones()`: This is a PyTorch function that creates a tensor filled with the value `1`.
# *   `vision_hidden.size()[:2]`: This determines the shape of the new `vision_mask` tensor. `vision_hidden` is a tensor representing the encoded visual features. `vision_hidden.size()` returns a tuple with its dimensions (e.g., `(batch_size, num_visual_features, hidden_size)`). `[:2]` takes the first two dimensions, which are typically `batch_size` and `num_visual_features`. So, the `vision_mask` will have a shape like `(batch_size, num_visual_features)`.
# *   `dtype=attention_mask.dtype`: This sets the data type of `vision_mask` to be the same as the `attention_mask` (which is typically for text). This ensures type compatibility when these masks are later concatenated.
# *   `device=attention_mask.device`: This places the newly created `vision_mask` tensor on the same computing device (e.g., CPU or GPU) as the `attention_mask` to facilitate operations on the same device.
#
# In summary, this line creates an attention mask for the visual features, where all elements are `1`, indicating that all visual features are active and should be attended to. This mask is consistent in shape, data type, and device with the attention mask used for text, allowing for their combination later in the fusion process.

In [ ]:
multimodal_model = MultimodalMBart(
    model
).to(device)

In [ ]:
input_ids = torch.randint(
    0,
    tokenizer.vocab_size,
    (2,12)
).to(device)


attention_mask = torch.ones(
    2,
    12
).to(device)

In [ ]:
image_features = torch.randn(
    2,
    36,
    1024
).to(device)

In [ ]:
labels = torch.randint(
    0,
    tokenizer.vocab_size,
    (2,15)
).to(device)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Indic-Multimodal-NMT"
)


METADATA_PATH = (
    PROJECT_ROOT /
    "data/processed/flickr30k_metadata.csv"
)


FEATURE_DIR = (
    PROJECT_ROOT /
    "data/features/detectron2"
)



In [ ]:
df = pd.read_csv(
    METADATA_PATH
)

df.head()

,image_id,filename,source_text
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...
1,0,1000092795.jpg,"Two young, White males are outside near many b..."
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.
4,0,1000092795.jpg,Two friends enjoy time spent together.


In [ ]:
class MultimodalDataset(Dataset):

    def __init__(
        self,
        dataframe,
        feature_dir,
        tokenizer,
        max_length=128
    ):

        self.df = dataframe.reset_index(drop=True)

        self.feature_dir = Path(feature_dir)

        self.tokenizer = tokenizer

        self.max_length = max_length


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]


        # -----------------------
        # English text
        # -----------------------

        text = row["source_text"]


        text_tokens = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )


        input_ids = (
            text_tokens["input_ids"]
            .squeeze(0)
        )


        attention_mask = (
            text_tokens["attention_mask"]
            .squeeze(0)
        )


        # -----------------------
        # Image features
        # -----------------------

        feature_path = (
            self.feature_dir /
            f"{Path(row['filename']).stem}.npy"
        )


        image_features = np.load(
            feature_path
        )


        image_features = torch.tensor(
            image_features,
            dtype=torch.float32
        )


        return {

            "input_ids": input_ids,

            "attention_mask": attention_mask,

            "image_features": image_features
        }

In [ ]:
feature_files = list(FEATURE_DIR.glob("*.npy"))

print("Features available:", len(feature_files))

Features available: 26702


In [ ]:
feature_files = {
    x.stem for x in FEATURE_DIR.glob("*.npy")
}

print("Features:", len(feature_files))


missing = df[
    ~df["filename"]
    .apply(lambda x: Path(x).stem in feature_files)
]

print("Missing samples:", len(missing))

Features: 26702
Missing samples: 21560


In [ ]:
available_features = {
    p.stem for p in FEATURE_DIR.glob("*.npy")
}

print("Available image features:", len(available_features))

Available image features: 26702


In [ ]:
df_train = df[
    df["filename"]
    .apply(lambda x: Path(x).stem in available_features)
].reset_index(drop=True)


print(df_train.shape)

(133510, 3)


In [ ]:
missing = []

for filename in df_train["filename"]:
    feature_file = FEATURE_DIR / f"{Path(filename).stem}.npy"

    if not feature_file.exists():
        missing.append(filename)


print("Missing:", len(missing))

Missing: 0


In [ ]:
dataset = MultimodalDataset(
    dataframe=df_train,
    feature_dir=FEATURE_DIR,
    tokenizer=tokenizer
)

In [ ]:
sample = dataset[0]

for key, value in sample.items():
    print(key, value.shape)

input_ids torch.Size([128])
attention_mask torch.Size([128])
image_features torch.Size([36, 1024])


In [ ]:
df_train.to_csv(
    PROJECT_ROOT / "data/processed/flickr30k_with_features.csv",
    index=False
)

In [ ]:
from torch.utils.data import Dataset
import torch
import numpy as np
from pathlib import Path


class MultimodalDataset(Dataset):

    def __init__(
        self,
        dataframe,
        feature_dir,
        tokenizer,
        max_length=128,
        train=True
    ):
        self.df = dataframe.reset_index(drop=True)
        self.feature_dir = Path(feature_dir)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        # -------------------------
        # English input
        # -------------------------

        self.tokenizer.src_lang = "en_XX"

        source = self.tokenizer(
            row["source_text"],
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # -------------------------
        # Detectron2 features
        # -------------------------

        feature_path = (
            self.feature_dir /
            f"{Path(row['filename']).stem}.npy"
        )

        image_features = np.load(feature_path)

        sample = {

            "input_ids":
                source["input_ids"].squeeze(0),

            "attention_mask":
                source["attention_mask"].squeeze(0),

            "image_features":
                torch.tensor(
                    image_features,
                    dtype=torch.float32
                )
        }

        # -------------------------
        # Hindi labels (if available)
        # -------------------------

        if (
            self.train and
            "target_text" in self.df.columns
        ):

            target = self.tokenizer(
                row["target_text"],
                max_length=self.max_length,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            sample["labels"] = target["input_ids"].squeeze(0)

        return sample

In [ ]:
dataset = MultimodalDataset(
    dataframe=df_train,
    feature_dir=FEATURE_DIR,
    tokenizer=tokenizer,
    train=False
)

In [ ]:
sample = dataset[0]

for k, v in sample.items():
    print(k, v.shape)

input_ids torch.Size([128])
attention_mask torch.Size([128])
image_features torch.Size([36, 1024])


In [ ]:
CHECKPOINT_FILE = PROJECT_ROOT / "checkpoints/translation_checkpoint.csv"
df_train = pd.read_csv(
    CHECKPOINT_FILE
)

dataset = MultimodalDataset(
    dataframe=df_train,
    feature_dir=FEATURE_DIR,
    tokenizer=tokenizer,
    train=True
)

In [ ]:
print(df_train.shape)
df_train.head()

(25840, 4)


,image_id,filename,source_text,target_text
0,0,1000092795.jpg,Two young guys with shaggy hair look at their ...,दो बूढ़े-बूढ़े बाल वाले लड़के yard में बैठे-बै...
1,0,1000092795.jpg,"Two young, White males are outside near many b...",दो युवा सफेद नर बाहर अनेक वृक्षों के पास रहते ...
2,0,1000092795.jpg,Two men in green shirts are standing in a yard.,हरे-भरे कमर पहने दो आदमी एक आँगन में खड़े हैं।
3,0,1000092795.jpg,A man in a blue shirt standing in a garden.,एक नीली शरीर पहने हुए आदमी जो एक बाग में खड़ा है।
4,0,1000092795.jpg,Two friends enjoy time spent together.,दो मित्रों को एक साथ बिताने का आनंद होता है।


In [ ]:
available_features = {
    p.stem for p in FEATURE_DIR.glob("*.npy")
}

df_train = df_train[
    df_train["filename"].apply(
        lambda x: Path(x).stem in available_features
    )
].reset_index(drop=True)

print(df_train.shape)

(25840, 4)


In [ ]:
sample = dataset[0]

for k, v in sample.items():
    print(k, v.shape)

input_ids torch.Size([128])
attention_mask torch.Size([128])
image_features torch.Size([36, 1024])
labels torch.Size([128])


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(len(train_loader))

3230


In [ ]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([8, 128])
attention_mask torch.Size([8, 128])
image_features torch.Size([8, 36, 1024])
labels torch.Size([8, 128])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

multimodal_model = multimodal_model.to(device)

batch = {
    k: v.to(device) if torch.is_tensor(v) else v
    for k, v in batch.items()
}

outputs = multimodal_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    image_features=batch["image_features"],
    labels=batch["labels"]
)

outputs = multimodal_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    image_features=batch["image_features"],
    labels=batch["labels"]
)

print(outputs.loss)

tensor(13.1472, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    multimodal_model.parameters(),
    lr=3e-5
)

In [ ]:
multimodal_model.train()

optimizer.zero_grad()

outputs = multimodal_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    image_features=batch["image_features"],
    labels=batch["labels"]
)

loss = outputs.loss

loss.backward()

optimizer.step()

print("Loss:", loss.item())

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.32 GiB is allocated by PyTorch, and 104.11 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

We put the model into training mode, clear any gradients left over from the previous batch, run a forward pass to get predictions and the loss, use loss.backward() to compute gradients for every trainable parameter, and finally call optimizer.step() to update the model's weights in the direction that reduces the loss.

In [ ]:
for name, param in multimodal_model.named_parameters():
    if "mbart" in name.lower():
        param.requires_grad = False


optimizer =  torch.optim.AdamW(
    filter(lambda p: p.requires_grad, multimodal_model.parameters()),
    lr=1e-4
)

multimodal_model.train()

for step, batch in enumerate(train_loader):
    if step >= 2: # remove for GPU, whole dataset training
        break

    batch = {
        k: v.to(device) if torch.is_tensor(v) else v
        for k, v in batch.items()
    }

    optimizer.zero_grad(set_to_none=True)

    outputs = multimodal_model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        image_features=batch["image_features"],
        labels=batch["labels"]
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    print(f"Step {step + 1}/2 | Loss: {loss.item():.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 229.81 MiB is free. Including non-PyTorch memory, this process has 14.34 GiB memory in use. Of the allocated memory 14.08 GiB is allocated by PyTorch, and 127.23 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
for name, param in multimodal_model.named_parameters():
    print(name, param.requires_grad)


total_params = sum(p.numel() for p in multimodal_model.parameters())
trainable_params = sum(
    p.numel()
    for p in multimodal_model.parameters()
    if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

mbart.model.shared.weight False
mbart.model.encoder.embed_positions.weight False
mbart.model.encoder.layers.0.self_attn.k_proj.weight False
mbart.model.encoder.layers.0.self_attn.k_proj.bias False
mbart.model.encoder.layers.0.self_attn.v_proj.weight False
mbart.model.encoder.layers.0.self_attn.v_proj.bias False
mbart.model.encoder.layers.0.self_attn.q_proj.weight False
mbart.model.encoder.layers.0.self_attn.q_proj.bias False
mbart.model.encoder.layers.0.self_attn.out_proj.weight False
mbart.model.encoder.layers.0.self_attn.out_proj.bias False
mbart.model.encoder.layers.0.self_attn_layer_norm.weight False
mbart.model.encoder.layers.0.self_attn_layer_norm.bias False
mbart.model.encoder.layers.0.fc1.weight False
mbart.model.encoder.layers.0.fc1.bias False
mbart.model.encoder.layers.0.fc2.weight False
mbart.model.encoder.layers.0.fc2.bias False
mbart.model.encoder.layers.0.final_layer_norm.weight False
mbart.model.encoder.layers.0.final_layer_norm.bias False
mbart.model.encoder.layers.1.se

In [ ]:
for name, param in multimodal_model.named_parameters():
    if "mbart" not in name.lower():
        print(name, param.shape, param.numel(), param.requires_grad)

vision_encoder.layer_norm.weight torch.Size([1024]) 1024 True
vision_encoder.layer_norm.bias torch.Size([1024]) 1024 True


In [ ]:
print(multimodal_model.fusion)

LateFusion()


In [ ]:
pip install sacrebleu

In [ ]:
# ==========================================
# BLEU EVALUATION
# ==========================================
import sacrebleu

multimodal_model.eval()

predictions = []
references = []

with torch.no_grad():

    for step, batch in enumerate(train_loader):

        if step >= 20:
           break

        batch = {
            k: v.to(device) if torch.is_tensor(v) else v
            for k, v in batch.items()
        }

        generated_ids = multimodal_model.generate(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            image_features=batch["image_features"],
            max_length=128,
            num_beams=1
        )

        # Model predictions
        preds = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Reference translations
        labels = batch["labels"].clone()

        labels[labels == -100] = tokenizer.pad_token_id

        refs = tokenizer.batch_decode(
            labels,
            skip_special_tokens=True
        )

        predictions.extend(preds)
        references.extend(refs)


# ==========================================
# SACREBLEU
# ==========================================

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

print(f"\nMultimodal BLEU: {bleu.score:.2f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Multimodal BLEU: 0.01
